
# 06 · FC-STEM (cepstral) — 약한 신호·혼합 비정질상 매핑

`RDF(structure factor) 방법`이 신호가 약해 잘 안 될 때의 대안. **EWPC(Exit-Wave Power Cepstrum)**:
$$\mathrm{EWPC}(\mathbf r)=\big|\,\mathcal F^{-1}\{\log I(\mathbf k)\}\,\big|$$
- **log** 이 다이내믹 레인지를 압축 → 약한 산란도 살아남음
- **배경 제거·structure factor 불필요** (실패하던 그 단계를 건너뜀)
- quefrency 축 = 실공간 거리(Å), 피크 = **원자간 거리** (RDF 유사)
- **Fluctuation(정규분산)** $F(R_p)=\langle C_p^2\rangle/\langle C_p\rangle^2-1$ 을 거리 밴드별로 → **혼합상 매핑**

> 참고: Pidaparthy, Ni, Hou, Abraham, Zuo, *Ultramicroscopy* **248** (2023) 113718.
> 켑스트럼은 모듈러스라 **비음수** → NMF도 유효(RDF의 부호 문제 없음).
> 모든 그림 PNG + 그래프 CSV는 각 셀에서 `SAVE_DIR`에 저장됩니다.


## 1) dm4 불러오기 & 설정

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/NBED/scan.dm4"   # ← STEM SI / 4D dm4 경로
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
Q_UNIT_HINT= "1/A"
Q_PER_PX   = 0.020        # 초기값. 실데이터는 메타데이터 사용(아래에서 갱신)
N_JOBS     = -2           # 병렬 코어수
EWPC_OFFSET= 1.0          # log(I+offset) — log(0) 방지
WINDOW     = True         # 켑스트럼 전 Hann 창(가장자리 십자 아티팩트 억제)
K          = 4            # 켑스트럼 프로파일 NMF/PCA 성분 수
CENTER     = None         # (cx,cy) 수동. None이면 무게중심
# 거리 밴드(Å) — 각각 하나의 FC-STEM 이미지. 데이터에 맞게 조절(먼저 §3 프로파일 보고).
BANDS      = [(1.5, 2.6), (2.6, 4.0), (4.0, 6.0), (1.5, 6.0)]

def make_fcstem_cube(scan=(36,48), dp=(96,96), empty_rows=8, seed=0):
    '''합성: 좌=결정(스팟), 중=비정질(halo), 우=다른 비정질(halo2), 아래=빈영역.
    FC-STEM이 결정/비정질/빈영역을 구분하는지 확인용.'''
    rng=np.random.default_rng(seed); Sy,Sx=scan; H,W=dp
    yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/(2*2.0**2))
    def halo(r0,s=4.0): return np.exp(-(rr-r0)**2/(2*s**2))
    def spots(r0,n=6,s=1.6,amp=3.0):
        img=np.zeros((H,W))
        for k in range(n):
            a=2*np.pi*k/n; sx,sy=cx+r0*np.cos(a),cy+r0*np.sin(a)
            img+=amp*np.exp(-((xx-sx)**2+(yy-sy)**2)/(2*s**2))
        return img
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            if iy>=Sy-empty_rows: base=0.9*beam
            elif ix<Sx//3:        base=beam+spots(22)+0.3*halo(22)   # 결정
            elif ix<2*Sx//3:      base=beam+1.2*halo(20)             # 비정질 A
            else:                 base=beam+1.2*halo(28)             # 비정질 B
            cube[iy,ix]=base+0.6*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube=fds.from_array(make_fcstem_cube(), q_per_px=Q_PER_PX, name="synthetic-FCSTEM")
else:
    cube=fds.load(DM4_PATH, Q_UNIT_HINT)
    print("raw loaded shape:", cube.data.shape, "(ndim", cube.ndim, ")")
    if cube.ndim<3:
        raise ValueError(f"{cube.ndim}D — not a scan; 로더가 데이터셋 검색 후에도 2D면 직접 로드하세요.")
    if DET_BIN>1: cube=fds.bin_cube_detector(cube, DET_BIN)
scan=cube.scan_shape; dp=cube.dp_shape
QPP = cube.calibration.q_per_px or Q_PER_PX
DR  = fds.quefrency_per_px(dp[0], QPP)      # 켑스트럼 픽셀당 Å
print("cube:", cube.shape, "| scan:", scan, "| dp:", dp)
print(f"q_per_px = {QPP:.5g} 1/A/px  ->  cepstral dr = {DR:.4g} A/px,  r_max ~ {DR*(dp[0]//2):.1f} A")

# 표시용 맵 재배열(3D 스택 대비) + 저장 헬퍼
import math
def _mapshape(n):
    r=int(math.sqrt(n))
    while r>1 and n%r: r-=1
    return (r,n//r) if r>1 else (1,n)
MAP=tuple(scan) if len(scan)==2 else _mapshape(int(np.prod(scan)))
def as_map(v):
    v=np.asarray(v,float).ravel(); m=np.full(int(np.prod(MAP)),np.nan); m[:min(v.size,m.size)]=v[:m.size]; return m.reshape(MAP)
SAVE_DIR=(os.path.dirname(DM4_PATH)+"/nb6_outputs" if not USE_SYNTHETIC else "nb6_outputs")
os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,name): p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name,header,rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:",p)
print("outputs ->", os.path.abspath(SAVE_DIR))


## 2) median 회절패턴 & 중심 (표시용; EWPC는 중심 위치에 둔감)

In [ ]:

med=fds.median_pattern(cube)
if CENTER is not None: cx,cy=CENTER
else: cx,cy=fds.center_of_mass(med, threshold=0.3)
print("center:", (round(cx,1),round(cy,1)))
fig,ax=plt.subplots(1,2,figsize=(9,4.2))
for a,d,t in [(ax[0],med,"median"),(ax[1],np.log1p(med),"median (log)")]:
    im=a.imshow(d,cmap="magma"); a.plot(cx,cy,"c+",ms=10); a.set_title(t); a.axis("off"); plt.colorbar(im,ax=a,fraction=0.046)
plt.tight_layout(); save(fig,"02_median"); plt.show()



## 3) 평균 EWPC + 켑스트럼 방사 프로파일 (원자간 거리)

전체 평균 켑스트럼과 그 방사 프로파일. **피크 = 원자간 거리(Å)**. 이게 약한 신호에서도 나오는 게 핵심.
아래 색 밴드가 §4 FC-STEM에 쓰는 거리 구간입니다. 피크가 물리적으로 타당한 Å(예: 2~4 Å)인지 확인하고,
어긋나면 `Q_UNIT_HINT`/`Q_PER_PX`(→ dr) 를 조정하세요.


In [ ]:

mcep=fds.ewpc_mean(cube, offset=EWPC_OFFSET, window=WINDOW, reducer="mean", n_jobs=N_JOBS, progress=True)
r_ax, prof=fds.cepstral_radial_profile(mcep, QPP, r_min=0.4)
n=mcep.shape[0]; ext=DR*(n//2)
fig,ax=plt.subplots(1,2,figsize=(12,4.4))
im=ax[0].imshow(mcep, cmap="inferno", extent=[-ext,ext,-ext,ext],
                vmax=np.percentile(mcep,99.5)); ax[0].set_title("mean EWPC (quefrency, Å)")
ax[0].set_xlabel("r_x (Å)"); ax[0].set_ylabel("r_y (Å)"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[1].plot(r_ax, prof, "k-")
for (ri,ro),c in zip(BANDS, plt.cm.viridis(np.linspace(0,0.85,len(BANDS)))):
    ax[1].axvspan(ri,ro,color=c,alpha=0.18)
ax[1].set_xlabel("quefrency r (Å)"); ax[1].set_ylabel("cepstral intensity")
ax[1].set_title("radial cepstral profile (peaks = interatomic distances)")
plt.tight_layout(); save(fig,"03_mean_ewpc_profile"); plt.show()
np.save(os.path.join(SAVE_DIR,"03_mean_ewpc.npy"), mcep)
save_csv("03_cepstral_radial_profile", ["r_A","cepstral_intensity"],
         [[f"{r_ax[i]:.4f}", f"{prof[i]:.6g}"] for i in range(len(r_ax))])



## 4) FC-STEM 이미지 — 거리 밴드별 fluctuation (혼합상 매핑)

각 거리 밴드에서 켑스트럼의 정규분산 $F(R_p)$ 를 스캔에 매핑. **밝음 = 큰 fluctuation**(그 거리 범위에서
질서/스펙클이 큼 → 결정질/특정 상). 밴드를 바꾸면 다른 상이 드러납니다.


In [ ]:

maps=fds.fluctuation_multiband(cube, BANDS, QPP, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)
nB=len(BANDS)
fig,ax=plt.subplots(1,nB,figsize=(3.2*nB,3.4),squeeze=False)
for j,((ri,ro),m) in enumerate(zip(BANDS,maps)):
    im=ax[0][j].imshow(as_map(m), cmap="viridis"); ax[0][j].set_title(f"F: {ri}-{ro} Å", fontsize=9)
    ax[0][j].axis("off"); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
fig.suptitle("FC-STEM fluctuation images (bright = ordered in that distance band)", y=1.03)
plt.tight_layout(); save(fig,"04_fcstem_fluctuation"); plt.show()
save_csv("04_fcstem_fluctuation", ["scan_index"]+[f"F_{ri}_{ro}A" for ri,ro in BANDS],
         [[i]+[f"{np.asarray(m).ravel()[i]:.6g}" for m in maps] for i in range(np.asarray(maps[0]).size)])



## 5) 켑스트럼 프로파일 분해 (NMF k=4) — 상 분리

각 위치의 켑스트럼 방사 프로파일(원자간 거리 시그니처)을 **NMF k=4** 로 분해 → 4개의 대표 프로파일과
위치별 분율맵. 켑스트럼은 **비음수**라 NMF가 그대로 유효합니다. (`method="pca"`로도 비교 가능)


In [ ]:

profs, r_p = fds.ewpc_profiles(cube, QPP, r_min=0.4, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)
# 비음수 보장(수치 잡음으로 음수면 0)
profs = np.clip(np.nan_to_num(profs), 0, None)
METHOD = "nmf"
dp_c = fds.decompose_profiles(profs, n_components=K, method=METHOD, x=r_p)
frac = dp_c.fractions
fig,ax=plt.subplots(2,K,figsize=(3.2*K,6))
for i in range(K):
    ax[0,i].plot(r_p, dp_c.components[i], lw=1.4); ax[0,i].set_title(f"comp {i+1}", fontsize=9)
    ax[0,i].set_xlabel("r (Å)")
    im=ax[1,i].imshow(as_map(frac[:,i]), cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9)
    ax[1,i].axis("off")
fig.suptitle(f"cepstral radial-profile {METHOD.upper()} (k={K})", y=1.02)
plt.tight_layout(); save(fig,"05_cepstral_profile_nmf"); plt.show()
save_csv("05_cepstral_components", ["r_A"]+[f"comp{i+1}" for i in range(K)],
         [[f"{r_p[j]:.4f}"]+[f"{dp_c.components[i][j]:.6g}" for i in range(K)] for j in range(len(r_p))])
print("\nAll outputs in:", os.path.abspath(SAVE_DIR))



**정리** — EWPC(로그→역FFT)로 배경 제거 없이 원자간 거리 신호를 얻고, (3) 평균 프로파일, (4) 거리 밴드별
FC-STEM fluctuation 매핑, (5) 프로파일 NMF k=4 로 혼합 비정질상을 분리합니다.
- **거리 밴드(`BANDS`)**: §3 프로파일에서 상별로 두드러지는 거리 구간을 골라 넣으세요.
- 위치가 많으면 (3)(4)(5)가 병렬로 수 분 걸릴 수 있습니다(`N_JOBS`, `DET_BIN`).
- 켑스트럼 거리축은 `dr=1/(N·q_per_px)`. 프로파일 피크가 예상 Å와 다르면 `Q_UNIT_HINT`/캘리브레이션 확인.
- 결정+비정질 혼합에 특히 강합니다(결정=밴드에서 밝음). 약한 리튬화합물 신호에 RDF 대안으로 권장.
